In [0]:
CREATE WIDGET TEXT catalog_name DEFAULT "workspace";

CREATE WIDGET TEXT source_file_name DEFAULT "creditcard.csv";

CREATE WIDGET TEXT batch_id DEFAULT "batch_001";

In [0]:
CREATE OR REPLACE TEMP VIEW silver_bronze_batch AS
SELECT bronze.*
FROM IDENTIFIER(:catalog_name || '.bronze.transactions_raw') AS bronze
WHERE bronze.batch_id = :batch_id
  AND bronze.source_file = :source_file_name
  AND EXISTS (
      SELECT 1
      FROM IDENTIFIER(:catalog_name || '.ops.processed_files') AS audit
      WHERE audit.source_s3_uri = bronze.source_s3_uri
        AND audit.batch_id = bronze.batch_id
        AND audit.status = 'SUCCESS'
  );


CREATE OR REPLACE TEMP VIEW silver_deduplicated_batch AS
WITH source_hashed AS (
    SELECT
        bronze.*,

        sha2(
            to_json(
                struct(
                    Time, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10,
                    V11, V12, V13, V14, V15, V16, V17, V18, V19, V20,
                    V21, V22, V23, V24, V25, V26, V27, V28,
                    Amount, Class
                )
            ),
            256
        ) AS record_hash

    FROM silver_bronze_batch AS bronze
),

ranked_rows AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY record_hash
            ORDER BY load_ts, pipeline_run_id
        ) AS duplicate_row_number

    FROM source_hashed
)

SELECT *
FROM ranked_rows
WHERE duplicate_row_number = 1;

In [0]:
CREATE OR REPLACE TEMP VIEW silver_validation_flags AS
SELECT
    bronze.*,

    CASE
        WHEN SIZE(
            FILTER(
                ARRAY(
                    CAST(Time AS STRING),
                    CAST(V1 AS STRING), CAST(V2 AS STRING), CAST(V3 AS STRING),
                    CAST(V4 AS STRING), CAST(V5 AS STRING), CAST(V6 AS STRING),
                    CAST(V7 AS STRING), CAST(V8 AS STRING), CAST(V9 AS STRING),
                    CAST(V10 AS STRING), CAST(V11 AS STRING), CAST(V12 AS STRING),
                    CAST(V13 AS STRING), CAST(V14 AS STRING), CAST(V15 AS STRING),
                    CAST(V16 AS STRING), CAST(V17 AS STRING), CAST(V18 AS STRING),
                    CAST(V19 AS STRING), CAST(V20 AS STRING), CAST(V21 AS STRING),
                    CAST(V22 AS STRING), CAST(V23 AS STRING), CAST(V24 AS STRING),
                    CAST(V25 AS STRING), CAST(V26 AS STRING), CAST(V27 AS STRING),
                    CAST(V28 AS STRING),
                    CAST(Amount AS STRING),
                    CAST(Class AS STRING)
                ),
                value -> value IS NULL
            )
        ) > 0 THEN TRUE
        ELSE FALSE
    END AS has_nulls,

    CASE WHEN Amount >= 0 THEN TRUE ELSE FALSE END AS is_valid_amount,

    CASE WHEN Class IN (0, 1) THEN TRUE ELSE FALSE END AS is_valid_class

FROM silver_deduplicated_batch AS bronze;


CREATE OR REPLACE TEMP VIEW silver_validated_batch AS
SELECT
    *,

    CASE
        WHEN has_nulls = FALSE
         AND is_valid_amount = TRUE
         AND is_valid_class = TRUE
        THEN 'VALID'
        ELSE 'INVALID'
    END AS validation_status,

    COALESCE(
        NULLIF(
            CONCAT_WS(
                '; ',
                CASE WHEN has_nulls THEN 'NULL_VALUE_PRESENT' END,
                CASE WHEN NOT is_valid_amount THEN 'NEGATIVE_AMOUNT' END,
                CASE WHEN NOT is_valid_class THEN 'INVALID_CLASS_VALUE' END
            ),
            ''
        ),
        'VALID'
    ) AS validation_reason

FROM silver_validation_flags;


CREATE OR REPLACE TEMP VIEW silver_standardized_batch AS
SELECT
    CAST(Time AS BIGINT) AS time,

    CAST(V1 AS DOUBLE) AS feature_01,
    CAST(V2 AS DOUBLE) AS feature_02,
    CAST(V3 AS DOUBLE) AS feature_03,
    CAST(V4 AS DOUBLE) AS feature_04,
    CAST(V5 AS DOUBLE) AS feature_05,
    CAST(V6 AS DOUBLE) AS feature_06,
    CAST(V7 AS DOUBLE) AS feature_07,
    CAST(V8 AS DOUBLE) AS feature_08,
    CAST(V9 AS DOUBLE) AS feature_09,
    CAST(V10 AS DOUBLE) AS feature_10,
    CAST(V11 AS DOUBLE) AS feature_11,
    CAST(V12 AS DOUBLE) AS feature_12,
    CAST(V13 AS DOUBLE) AS feature_13,
    CAST(V14 AS DOUBLE) AS feature_14,
    CAST(V15 AS DOUBLE) AS feature_15,
    CAST(V16 AS DOUBLE) AS feature_16,
    CAST(V17 AS DOUBLE) AS feature_17,
    CAST(V18 AS DOUBLE) AS feature_18,
    CAST(V19 AS DOUBLE) AS feature_19,
    CAST(V20 AS DOUBLE) AS feature_20,
    CAST(V21 AS DOUBLE) AS feature_21,
    CAST(V22 AS DOUBLE) AS feature_22,
    CAST(V23 AS DOUBLE) AS feature_23,
    CAST(V24 AS DOUBLE) AS feature_24,
    CAST(V25 AS DOUBLE) AS feature_25,
    CAST(V26 AS DOUBLE) AS feature_26,
    CAST(V27 AS DOUBLE) AS feature_27,
    CAST(V28 AS DOUBLE) AS feature_28,

    CAST(Amount AS DECIMAL(18,2)) AS amount,
    CAST(Class AS INT) AS class,

    CASE
        WHEN Class = 1 THEN 'Fraud'
        ELSE 'Legitimate'
    END AS fraud_label,

    record_hash,
    CONCAT('txn_', record_hash) AS transaction_id,

    has_nulls,
    is_valid_amount,
    is_valid_class,
    validation_status,
    validation_reason,

    load_ts,
    source_file,
    source_s3_uri,
    batch_id,
    ingestion_date,
    pipeline_run_id

FROM silver_validated_batch
WHERE validation_status = 'VALID';

In [0]:
CREATE OR REPLACE TEMP VIEW silver_time_amount_features AS
WITH timestamped AS (
    SELECT
        standardized.*,

        timestampadd(
            SECOND,
            time,
            TIMESTAMP('2026-08-01 00:00:00')
        ) AS transaction_timestamp

    FROM silver_standardized_batch AS standardized
),

time_features AS (
    SELECT
        *,

        CAST(transaction_timestamp AS DATE) AS transaction_date,
        HOUR(transaction_timestamp) AS transaction_hour,

        CASE
            WHEN HOUR(transaction_timestamp) BETWEEN 5 AND 11 THEN 'Morning'
            WHEN HOUR(transaction_timestamp) BETWEEN 12 AND 16 THEN 'Afternoon'
            WHEN HOUR(transaction_timestamp) BETWEEN 17 AND 20 THEN 'Evening'
            ELSE 'Night'
        END AS time_of_day,

        CASE
            WHEN HOUR(transaction_timestamp) BETWEEN 9 AND 17
            THEN 'Business Hours'
            ELSE 'Off Hours'
        END AS time_band

    FROM timestamped
),

amount_features AS (
    SELECT
        *,

        CASE
            WHEN amount < 10 THEN 'Very Low'
            WHEN amount < 50 THEN 'Low'
            WHEN amount < 100 THEN 'Medium'
            WHEN amount < 500 THEN 'High'
            ELSE 'Very High'
        END AS amount_bucket,

        ROUND(PERCENT_RANK() OVER (ORDER BY amount), 6) AS amount_percentile,

        ROUND(
            (amount - AVG(amount) OVER ())
            / NULLIF(STDDEV(amount) OVER (), 0),
            6
        ) AS amount_z_score

    FROM time_features
)

SELECT
    *,

    CASE
        WHEN ABS(amount_z_score) >= 3 THEN TRUE
        ELSE FALSE
    END AS amount_outlier_flag

FROM amount_features;

In [0]:
CREATE OR REPLACE TEMP VIEW silver_activity_features AS
WITH sequenced AS (
    SELECT
        features.*,

        LAG(transaction_timestamp) OVER (
            ORDER BY transaction_timestamp, transaction_id
        ) AS previous_transaction_timestamp,

        LAG(amount) OVER (
            ORDER BY transaction_timestamp, transaction_id
        ) AS previous_amount,

        sha2(
            concat_ws(
                '||',
                CAST(amount AS STRING),
                CAST(feature_01 AS STRING),
                CAST(feature_02 AS STRING),
                CAST(feature_03 AS STRING),
                CAST(feature_04 AS STRING),
                CAST(feature_05 AS STRING)
            ),
            256
        ) AS anonymized_pattern_key

    FROM silver_time_amount_features AS features
),

pattern_window AS (
    SELECT
        *,

        COUNT(*) OVER (
            PARTITION BY anonymized_pattern_key
            ORDER BY transaction_timestamp
            RANGE BETWEEN INTERVAL 5 MINUTES PRECEDING AND CURRENT ROW
        ) AS repeated_anonymized_pattern_count

    FROM sequenced
)

SELECT
    *,

    CASE
        WHEN repeated_anonymized_pattern_count > 1 THEN TRUE
        ELSE FALSE
    END AS repeated_anonymized_pattern_flag,

    CASE
        WHEN previous_transaction_timestamp IS NOT NULL
         AND timestampdiff(
                SECOND,
                previous_transaction_timestamp,
                transaction_timestamp
             ) <= 10
         AND amount = previous_amount
        THEN TRUE
        ELSE FALSE
    END AS rapid_repeat_flag

FROM pattern_window;

In [0]:
CREATE OR REPLACE TEMP VIEW silver_final_batch AS
WITH risk_scored AS (
    SELECT
        activity.*,

        (
            CASE WHEN rapid_repeat_flag THEN 3 ELSE 0 END
            +
            CASE WHEN repeated_anonymized_pattern_flag THEN 2 ELSE 0 END
            +
            CASE WHEN amount_bucket IN ('Very High', 'Very Low') THEN 2 ELSE 0 END
            +
            CASE WHEN amount_outlier_flag THEN 1 ELSE 0 END
            +
            CASE
                WHEN amount_bucket IN ('Very High', 'Very Low')
                 AND time_of_day = 'Night'
                THEN 1
                WHEN rapid_repeat_flag
                 AND time_band = 'Business Hours'
                THEN 1
                ELSE 0
            END
        ) AS review_score

    FROM silver_activity_features AS activity
),

risk_classified AS (
    SELECT
        *,

        CASE
            WHEN review_score >= 5 THEN 'High'
            WHEN review_score >= 3 THEN 'Medium'
            ELSE 'Low'
        END AS review_priority,

        COALESCE(
            NULLIF(
                CONCAT_WS(
                    '; ',
                    CASE WHEN rapid_repeat_flag
                         THEN 'Rapid repeated transaction' END,
                    CASE WHEN repeated_anonymized_pattern_flag
                         THEN 'Repeated anonymized feature pattern' END,
                    CASE WHEN amount_bucket IN ('Very High', 'Very Low')
                         THEN 'Extreme amount bucket' END,
                    CASE WHEN amount_outlier_flag
                         THEN 'Amount outlier' END,
                    CASE
                        WHEN amount_bucket IN ('Very High', 'Very Low')
                         AND time_of_day = 'Night'
                        THEN 'Night-time context paired with extreme amount'
                    END,
                    CASE
                        WHEN rapid_repeat_flag
                         AND time_band = 'Business Hours'
                        THEN 'Rapid repeat during business hours'
                    END
                ),
                ''
            ),
            'No review indicators'
        ) AS risk_reason

    FROM risk_scored
)

SELECT
    transaction_id,
    transaction_timestamp,
    transaction_date,
    transaction_hour,
    time_of_day,
    time_band,

    time,
    feature_01,
    feature_02,
    feature_03,
    feature_04,
    feature_05,
    feature_06,
    feature_07,
    feature_08,
    feature_09,
    feature_10,
    feature_11,
    feature_12,
    feature_13,
    feature_14,
    feature_15,
    feature_16,
    feature_17,
    feature_18,
    feature_19,
    feature_20,
    feature_21,
    feature_22,
    feature_23,
    feature_24,
    feature_25,
    feature_26,
    feature_27,
    feature_28,

    amount,
    amount_bucket,
    amount_z_score,
    amount_percentile,
    amount_outlier_flag,

    class,
    fraud_label,

    has_nulls,
    is_valid_amount,
    is_valid_class,
    validation_status,
    validation_reason,

    rapid_repeat_flag,
    repeated_anonymized_pattern_flag,

    review_score,
    review_priority,
    risk_reason,

    record_hash,

    load_ts,
    source_file,
    source_s3_uri,
    batch_id,
    ingestion_date,
    pipeline_run_id

FROM risk_classified;

In [0]:
CREATE TABLE IF NOT EXISTS
IDENTIFIER(:catalog_name || '.silver.transactions_silver_incremental')
USING DELTA
AS
SELECT *
FROM silver_final_batch
WHERE 1 = 0;


CREATE TABLE IF NOT EXISTS
IDENTIFIER(:catalog_name || '.ops.silver_processed_batches') (
    source_file_name STRING,
    source_s3_uri STRING,
    batch_id STRING,
    pipeline_run_id STRING,
    status STRING,

    bronze_row_count BIGINT,
    duplicate_rows_removed BIGINT,
    invalid_rows_removed BIGINT,
    silver_row_count BIGINT,

    started_ts TIMESTAMP,
    completed_ts TIMESTAMP,
    error_message STRING,
    target_table STRING
)
USING DELTA;

In [0]:
INSERT INTO IDENTIFIER(:catalog_name || '.ops.silver_processed_batches')
SELECT
    (
        SELECT MAX(source_file)
        FROM silver_bronze_batch
    ) AS source_file_name,

    (
        SELECT MAX(source_s3_uri)
        FROM silver_bronze_batch
    ) AS source_s3_uri,

    :batch_id AS batch_id,

    (
        SELECT MAX(pipeline_run_id)
        FROM silver_bronze_batch
    ) AS pipeline_run_id,

    'RUNNING' AS status,

    (
        SELECT COUNT(*)
        FROM silver_bronze_batch
    ) AS bronze_row_count,

    (
        SELECT COUNT(*)
        FROM silver_bronze_batch
    )
    -
    (
        SELECT COUNT(*)
        FROM silver_deduplicated_batch
    ) AS duplicate_rows_removed,

    (
        SELECT COUNT(*)
        FROM silver_deduplicated_batch
    )
    -
    (
        SELECT COUNT(*)
        FROM silver_final_batch
    ) AS invalid_rows_removed,

    NULL AS silver_row_count,
    current_timestamp() AS started_ts,
    NULL AS completed_ts,
    NULL AS error_message,

    :catalog_name || '.silver.transactions_silver_incremental' AS target_table

FROM (SELECT 1 AS audit_seed)

WHERE EXISTS (
    SELECT 1
    FROM silver_bronze_batch
)

AND NOT EXISTS (
    SELECT 1
    FROM IDENTIFIER(:catalog_name || '.ops.silver_processed_batches')
    WHERE batch_id = :batch_id
      AND source_file_name = :source_file_name
      AND status IN ('RUNNING', 'SUCCESS')
);

num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO IDENTIFIER(:catalog_name || '.silver.transactions_silver_incremental')
SELECT *
FROM silver_final_batch
WHERE NOT EXISTS (
    SELECT 1
    FROM IDENTIFIER(:catalog_name || '.silver.transactions_silver_incremental')
    WHERE batch_id = :batch_id
      AND source_file = :source_file_name
);


UPDATE IDENTIFIER(:catalog_name || '.ops.silver_processed_batches')
SET
    status = 'SUCCESS',

    silver_row_count = (
        SELECT COUNT(*)
        FROM IDENTIFIER(:catalog_name || '.silver.transactions_silver_incremental')
        WHERE batch_id = :batch_id
          AND source_file = :source_file_name
    ),

    completed_ts = current_timestamp(),
    error_message = NULL

WHERE batch_id = :batch_id
  AND source_file_name = :source_file_name
  AND status = 'RUNNING';

num_affected_rows
1
